![image.png](https://i.imgur.com/a3uAqnb.png)

# Text Classification with Neural Networks - Homework Assignment

In this homework, you will implement multiple **neural network architectures for text classification** to detect sarcasm in news headlines. This project will help you understand the fundamentals of text preprocessing, feature extraction, and various neural network approaches for NLP tasks.

## 📌 Project Overview
- **Task**: Classify news headlines as sarcastic or non-sarcastic
- **Architectures**: Naive Bayes, 2-Layer NN, RNN, GRU, and LSTM
- **Dataset**: News Headlines Dataset for Sarcasm Detection
- **Goal**: Compare different approaches for text classification

## 📚 Learning Objectives
By completing this assignment, you will:
- Understand text preprocessing and feature extraction techniques
- Learn TF-IDF vectorization for text data
- Implement multiple neural network architectures for text classification
- Compare traditional ML with deep learning approaches
- Practice evaluation metrics for classification tasks

## 1️⃣ Initial Setup and Data Download

**Task**: Download the sarcasm detection dataset and explore its structure.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rmisra/news-headlines-dataset-for-sarcasm-detection")

print("Path to dataset files:", path)

In [ ]:
import os
os.listdir(path)

## 2️⃣ Data Loading and Exploration

**Task**: Load the dataset and perform initial exploration to understand the data structure and characteristics.

**Requirements**:
- Load the JSON dataset
- Convert to DataFrame format
- Explore class distribution
- Display sample headlines from both classes
- Analyze basic statistics about headline lengths

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

with open(f'{path}/Sarcasm_Headlines_Dataset_v2.json', 'r') as f:
    data = [json.loads(line) for line in f]

df = pd.DataFrame(data)


class_proportions = df['is_sarcastic'].value_counts(normalize=True)

In [ ]:
print("\n" + "="*50)
print("EXAMPLES OF SARCASTIC HEADLINES:")
print("="*50)
sarcastic_examples = df[df['is_sarcastic'] == 1]['headline'].head(5)
for i, headline in enumerate(sarcastic_examples, 1):
    print(f"{i}. {headline}")

print("\n" + "="*50)
print("EXAMPLES OF NON-SARCASTIC HEADLINES:")
print("="*50)
non_sarcastic_examples = df[df['is_sarcastic'] == 0]['headline'].head(5)
for i, headline in enumerate(non_sarcastic_examples, 1):
    print(f"{i}. {headline}")

In [ ]:
df['headline_length'] = df['headline'].str.len()
df['word_count'] = df['headline'].str.split().str.len()

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
df['is_sarcastic'].value_counts().plot(kind='bar')
plt.title('Class Distribution')
plt.xlabel('Is Sarcastic (0=No, 1=Yes)')
plt.ylabel('Count')
plt.xticks(rotation=0)

plt.subplot(1, 2, 2)
plt.hist([df[df['is_sarcastic']==0]['word_count'], df[df['is_sarcastic']==1]['word_count']], 
         bins=30, alpha=0.7, label=['Non-Sarcastic', 'Sarcastic'])
plt.title('Word Count Distribution by Class')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
df

## 3️⃣ Text Preprocessing and Feature Engineering

**Task**: Implement comprehensive text preprocessing pipeline for neural network training.

**Requirements**:
- Download required NLTK data
- Implement text cleaning and preprocessing function
- Apply preprocessing to all headlines
- Create train-test split with stratification

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import string

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab',quiet=True)

In [ ]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """Comprehensive text preprocessing function"""
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs, mentions, hashtags
    text = re.sub(r'http\S+|www\S+|@\w+|#\w+', '', text)
    
    # Remove punctuation and numbers
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords and short words, apply stemming
    tokens = [stemmer.stem(word) for word in tokens 
              if word not in stop_words and len(word) > 2]
    
    return ' '.join(tokens)

print("Preprocessing headlines...")
df['processed_headline'] = df['headline'].apply(preprocess_text)

df = df[df['processed_headline'].str.len() > 0].reset_index(drop=True)

X = df['processed_headline']
y = df['is_sarcastic']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Dataset after preprocessing: {len(df)} samples")
print(f"Train set: {len(X_train)} samples | Test set: {len(X_test)} samples")
print(f"Train class distribution: {y_train.value_counts().to_dict()}")

In [ ]:
print("\nPreprocessing Examples:")
for i in range(3):
    print(f"Original: {df.iloc[i]['headline']}")
    print(f"Processed: {df.iloc[i]['processed_headline']}")
    print(f"Label: {df.iloc[i]['is_sarcastic']}\n")

## 4️⃣ TF-IDF Vectorization and Naive Bayes Baseline

**Task**: Create TF-IDF feature matrices and implement a Naive Bayes baseline model.

**Requirements**:
- Create TF-IDF vectorizer with appropriate parameters
- Transform text data into numerical features
- Train and evaluate Naive Bayes classifier
- Store results for final comparison

In [ ]:
# Cell 3: TF-IDF and Naive Bayes Model
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Create TF-IDF matrix
print("Creating TF-IDF matrices...")
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,  # Top 10k features
    min_df=2,            # Ignore terms that appear in less than 2 documents
    max_df=0.95,         # Ignore terms that appear in more than 95% of documents
    ngram_range=(1, 2)   # Use unigrams and bigrams
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"TF-IDF matrix shape - Train: {X_train_tfidf.shape}, Test: {X_test_tfidf.shape}")

# Train Naive Bayes model
print("\nTraining Naive Bayes model...")
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_train_tfidf, y_train)

nb_train_pred = nb_model.predict(X_train_tfidf)
nb_test_pred = nb_model.predict(X_test_tfidf)

nb_train_acc = accuracy_score(y_train, nb_train_pred)
nb_test_acc = accuracy_score(y_test, nb_test_pred)

print(f"Naive Bayes - Train Accuracy: {nb_train_acc:.4f}")
print(f"Naive Bayes - Test Accuracy: {nb_test_acc:.4f}")

results = {
    'Naive Bayes': {
        'train_acc': nb_train_acc,
        'test_acc': nb_test_acc,
        'predictions': nb_test_pred,
        'model': nb_model
    }
}

print("\nNaive Bayes Classification Report:")
print(classification_report(y_test, nb_test_pred, target_names=['Non-Sarcastic', 'Sarcastic']))

## 5️⃣ 2-Layer Neural Network on TF-IDF Features

**Task**: Implement a 2-layer neural network using TF-IDF features as input.

**Requirements**:
- Convert TF-IDF matrices to PyTorch tensors
- Define 2-layer neural network architecture
- Implement training loop with proper batch processing
- Evaluate model performance

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score
import numpy as np

X_train_tensor = torch.FloatTensor(X_train_tfidf.toarray())
X_test_tensor = torch.FloatTensor(X_test_tfidf.toarray())
y_train_tensor = torch.LongTensor(y_train.values)
y_test_tensor = torch.LongTensor(y_test.values)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

class TwoLayerNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_classes=2):
        super(TwoLayerNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TwoLayerNN(input_size=X_train_tensor.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Training on device: {device}")
print(f"Model architecture: {X_train_tensor.shape[1]} -> 128 -> 2")

num_epochs = 20
model.train()
for epoch in range(num_epochs):
    total_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}')

def evaluate_model(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())
    
    return np.array(all_preds), np.array(all_labels)

train_preds, train_labels = evaluate_model(model, train_loader, device)
test_preds, test_labels = evaluate_model(model, test_loader, device)

nn_train_acc = accuracy_score(train_labels, train_preds)
nn_test_acc = accuracy_score(test_labels, test_preds)

print(f"\n2-Layer NN - Train Accuracy: {nn_train_acc:.4f}")
print(f"2-Layer NN - Test Accuracy: {nn_test_acc:.4f}")

results['2-Layer NN'] = {
    'train_acc': nn_train_acc,
    'test_acc': nn_test_acc,
    'predictions': test_preds,
    'model': model
}

print("\n2-Layer NN Classification Report:")
print(classification_report(test_labels, test_preds, target_names=['Non-Sarcastic', 'Sarcastic']))

## 6️⃣ Text Tokenization for RNN Models

**Task**: Prepare text data for RNN-based models by building vocabulary and converting text to sequences.

**Requirements**:
- Build vocabulary from training data with frequency threshold
- Convert text to sequences of vocabulary indices
- Pad sequences to uniform length
- Create DataLoaders for RNN training

In [ ]:
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

def build_vocab(texts, min_freq=2):
    """Build vocabulary from texts with minimum frequency threshold"""
    word_counts = Counter()
    for text in texts:
        words = text.split()
        word_counts.update(words)
    
    # Create word to index mapping
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, count in word_counts.items():
        if count >= min_freq:
            vocab[word] = len(vocab)
    
    return vocab

vocab = build_vocab(X_train, min_freq=2)
vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")

def text_to_sequence(text, vocab, max_length=50):
    """Convert text to sequence of vocabulary indices"""
    words = text.split()[:max_length]  # Truncate if too long
    sequence = [vocab.get(word, vocab['<UNK>']) for word in words]
    return sequence

max_seq_length = 50
X_train_sequences = [text_to_sequence(text, vocab, max_seq_length) for text in X_train]
X_test_sequences = [text_to_sequence(text, vocab, max_seq_length) for text in X_test]

X_train_padded = pad_sequence([torch.LongTensor(seq) for seq in X_train_sequences], 
                              batch_first=True, padding_value=0)
X_test_padded = pad_sequence([torch.LongTensor(seq) for seq in X_test_sequences], 
                             batch_first=True, padding_value=0)

max_len = max(X_train_padded.size(1), X_test_padded.size(1))
if X_train_padded.size(1) < max_len:
    padding = torch.zeros(X_train_padded.size(0), max_len - X_train_padded.size(1), dtype=torch.long)
    X_train_padded = torch.cat([X_train_padded, padding], dim=1)
if X_test_padded.size(1) < max_len:
    padding = torch.zeros(X_test_padded.size(0), max_len - X_test_padded.size(1), dtype=torch.long)
    X_test_padded = torch.cat([X_test_padded, padding], dim=1)

print(f"Tokenized sequences shape - Train: {X_train_padded.shape}, Test: {X_test_padded.shape}")

# Create data loaders for RNN models
rnn_train_dataset = TensorDataset(X_train_padded, y_train_tensor)
rnn_test_dataset = TensorDataset(X_test_padded, y_test_tensor)
rnn_train_loader = DataLoader(rnn_train_dataset, batch_size=64, shuffle=True)
rnn_test_loader = DataLoader(rnn_test_dataset, batch_size=64, shuffle=False)


In [ ]:
print("\nTokenization Examples:")
for i in range(3):
    original = X_train.iloc[i]
    tokenized = X_train_sequences[i][:10]  # Show first 10 tokens
    words = [list(vocab.keys())[list(vocab.values()).index(idx)] for idx in tokenized]
    print(f"Original: {original}")
    print(f"Tokens: {tokenized}")
    print(f"Words: {words}\n")

print(f"Ready for RNN training! Vocab size: {vocab_size}, Max sequence length: {max_len}")

## 7️⃣ Recurrent Neural Network (RNN) Implementation

**Task**: Implement a Recurrent Neural Network for text classification using word embeddings.

**Requirements**:
- Define RNN architecture with embedding layer
- Implement proper masking for padded sequences
- Use average pooling over sequence length
- Train and evaluate the RNN model

In [ ]:
# Cell 6: RNN Model
import torch.nn as nn

# Define RNN model
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_classes=2, num_layers=2):
        super(RNNClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers=num_layers, 
                         batch_first=True, dropout=0.3)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x):
        mask = (x != 0).float() 
        
        embedded = self.embedding(x)  
        rnn_out, hidden = self.rnn(embedded)  
        
        masked_output = rnn_out * mask.unsqueeze(-1)
        seq_lengths = mask.sum(dim=1, keepdim=True)
        seq_lengths = torch.clamp(seq_lengths, min=1)  
        
        pooled = masked_output.sum(dim=1) / seq_lengths
        pooled = self.dropout(pooled)
        output = self.fc(pooled)
        return output

rnn_model = RNNClassifier(vocab_size=vocab_size).to(device)
rnn_criterion = nn.CrossEntropyLoss()
rnn_optimizer = optim.Adam(rnn_model.parameters(), lr=0.003, weight_decay=1e-4)

print(f"RNN Model Architecture:")
print(f"Embedding: {vocab_size} -> 128")
print(f"RNN: 128 -> 256 (2 layers)")
print(f"Output: 256 -> 2")

def train_rnn_model(model, train_loader, criterion, optimizer, num_epochs=20):
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        correct = 0
        total = 0
        
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
        
        if (epoch + 1) % 5 == 0:
            epoch_acc = 100 * correct / total
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}, Train Acc: {epoch_acc:.2f}%')

print("\nTraining RNN model...")
train_rnn_model(rnn_model, rnn_train_loader, rnn_criterion, rnn_optimizer, num_epochs=20)

def evaluate_rnn_model(model, train_loader, test_loader, device):
    model.eval()
    
    def get_predictions(loader):
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for batch_x, batch_y in loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                outputs = model(batch_x)
                _, predicted = torch.max(outputs.data, 1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(batch_y.cpu().numpy())
        return np.array(all_preds), np.array(all_labels)
    
    train_preds, train_labels = get_predictions(train_loader)
    test_preds, test_labels = get_predictions(test_loader)
    
    return train_preds, train_labels, test_preds, test_labels

train_preds, train_labels, test_preds, test_labels = evaluate_rnn_model(
    rnn_model, rnn_train_loader, rnn_test_loader, device)

rnn_train_acc = accuracy_score(train_labels, train_preds)
rnn_test_acc = accuracy_score(test_labels, test_preds)

print(f"\nRNN - Train Accuracy: {rnn_train_acc:.4f}")
print(f"RNN - Test Accuracy: {rnn_test_acc:.4f}")

results['RNN'] = {
    'train_acc': rnn_train_acc,
    'test_acc': rnn_test_acc,
    'predictions': test_preds,
    'model': rnn_model
}

print("\nRNN Classification Report:")
print(classification_report(test_labels, test_preds, target_names=['Non-Sarcastic', 'Sarcastic']))

## 8️⃣ Gated Recurrent Unit (GRU) Implementation

**Task**: Implement a bidirectional GRU model for improved sequence modeling.

**Requirements**:
- Define GRU architecture with bidirectional processing
- Handle doubled hidden dimension from bidirectional GRU
- Apply same training and evaluation procedures
- Compare performance with basic RNN

In [ ]:
import torch.nn as nn

class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_classes=2, num_layers=2):
        super(GRUClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, 
                         batch_first=True, dropout=0.3, bidirectional=True)
        self.dropout = nn.Dropout(0.5)
        # Note: bidirectional GRU doubles the hidden dimension
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        
    def forward(self, x):
        mask = (x != 0).float() 
        
        embedded = self.embedding(x)  
        gru_out, hidden = self.gru(embedded)  
        
        masked_output = gru_out * mask.unsqueeze(-1)
        seq_lengths = mask.sum(dim=1, keepdim=True)
        seq_lengths = torch.clamp(seq_lengths, min=1) 
        
        pooled = masked_output.sum(dim=1) / seq_lengths
        pooled = self.dropout(pooled)
        output = self.fc(pooled)
        return output

gru_model = GRUClassifier(vocab_size=vocab_size).to(device)
gru_criterion = nn.CrossEntropyLoss()
gru_optimizer = optim.Adam(gru_model.parameters(), lr=0.003, weight_decay=1e-4)

print(f"GRU Model Architecture:")
print(f"Embedding: {vocab_size} -> 128")
print(f"Bidirectional GRU: 128 -> 256*2 (2 layers)")
print(f"Output: 512 -> 2")

print("\nTraining GRU model...")
train_rnn_model(gru_model, rnn_train_loader, gru_criterion, gru_optimizer, num_epochs=20)

train_preds, train_labels, test_preds, test_labels = evaluate_rnn_model(
    gru_model, rnn_train_loader, rnn_test_loader, device)

gru_train_acc = accuracy_score(train_labels, train_preds)
gru_test_acc = accuracy_score(test_labels, test_preds)

print(f"\nGRU - Train Accuracy: {gru_train_acc:.4f}")
print(f"GRU - Test Accuracy: {gru_test_acc:.4f}")

results['GRU'] = {
    'train_acc': gru_train_acc,
    'test_acc': gru_test_acc,
    'predictions': test_preds,
    'model': gru_model
}

print("\nGRU Classification Report:")
print(classification_report(test_labels, test_preds, target_names=['Non-Sarcastic', 'Sarcastic']))

## 9️⃣ Long Short-Term Memory (LSTM) Implementation

**Task**: Implement a bidirectional LSTM model for capturing long-term dependencies in text.

**Requirements**:
- Define LSTM architecture with bidirectional processing
- Handle both hidden and cell states from LSTM
- Apply consistent training methodology
- Compare with RNN and GRU performance

In [ ]:
# Cell 8: LSTM Model
import torch.nn as nn

# Define LSTM model
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_classes=2, num_layers=2):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, 
                           batch_first=True, dropout=0.3, bidirectional=True)
        self.dropout = nn.Dropout(0.5)
        # Note: bidirectional LSTM doubles the hidden dimension
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        
    def forward(self, x):
        mask = (x != 0).float() 
        

        embedded = self.embedding(x)  
        lstm_out, (hidden, cell) = self.lstm(embedded) 
        
        masked_output = lstm_out * mask.unsqueeze(-1)
        seq_lengths = mask.sum(dim=1, keepdim=True)
        seq_lengths = torch.clamp(seq_lengths, min=1) 
        
        pooled = masked_output.sum(dim=1) / seq_lengths
        pooled = self.dropout(pooled)
        output = self.fc(pooled)
        return output

lstm_model = LSTMClassifier(vocab_size=vocab_size).to(device)
lstm_criterion = nn.CrossEntropyLoss()
lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=0.003, weight_decay=1e-4)

print(f"LSTM Model Architecture:")
print(f"Embedding: {vocab_size} -> 128")
print(f"Bidirectional LSTM: 128 -> 256*2 (2 layers)")
print(f"Output: 512 -> 2")

print("\nTraining LSTM model...")
train_rnn_model(lstm_model, rnn_train_loader, lstm_criterion, lstm_optimizer, num_epochs=20)

train_preds, train_labels, test_preds, test_labels = evaluate_rnn_model(
    lstm_model, rnn_train_loader, rnn_test_loader, device)

lstm_train_acc = accuracy_score(train_labels, train_preds)
lstm_test_acc = accuracy_score(test_labels, test_preds)

print(f"\nLSTM - Train Accuracy: {lstm_train_acc:.4f}")
print(f"LSTM - Test Accuracy: {lstm_test_acc:.4f}")

results['LSTM'] = {
    'train_acc': lstm_train_acc,
    'test_acc': lstm_test_acc,
    'predictions': test_preds,
    'model': lstm_model
}

print("\nLSTM Classification Report:")
print(classification_report(test_labels, test_preds, target_names=['Non-Sarcastic', 'Sarcastic']))

## 🔟 Comprehensive Model Comparison and Analysis

**Task**: Compare all implemented models and analyze their performance through visualizations and metrics.

**Requirements**:
- Create comprehensive results summary comparing all models
- Generate confusion matrices for each model
- Visualize training progress and model rankings
- Analyze overfitting and generalization performance
- Provide detailed performance breakdown

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import pandas as pd

print("="*80)
print("FINAL MODEL COMPARISON - SARCASM DETECTION")
print("="*80)

model_names = list(results.keys())
train_accuracies = [results[model]['train_acc'] for model in model_names]
test_accuracies = [results[model]['test_acc'] for model in model_names]

comparison_df = pd.DataFrame({
    'Model': model_names,
    'Train Accuracy': train_accuracies,
    'Test Accuracy': test_accuracies,
    'Overfitting (Train-Test)': [train - test for train, test in zip(train_accuracies, test_accuracies)]
})

print(comparison_df.round(4))

best_model_idx = comparison_df['Test Accuracy'].idxmax()
best_model_name = comparison_df.loc[best_model_idx, 'Model']
best_test_acc = comparison_df.loc[best_model_idx, 'Test Accuracy']

print(f"\nBest performing model: {best_model_name} with {best_test_acc:.4f} test accuracy")

fig, axes = plt.subplots(3, 3, figsize=(18, 18))
fig.suptitle('Model Performance Comparison and Confusion Matrices', fontsize=16)

ax1 = axes[0, 0]
x = range(len(model_names))
width = 0.35
ax1.bar([i - width/2 for i in x], train_accuracies, width, label='Train Accuracy', alpha=0.8)
ax1.bar([i + width/2 for i in x], test_accuracies, width, label='Test Accuracy', alpha=0.8)
ax1.set_xlabel('Models')
ax1.set_ylabel('Accuracy')
ax1.set_title('Train vs Test Accuracy')
ax1.set_xticks(x)
ax1.set_xticklabels(model_names, rotation=45)
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
overfitting = comparison_df['Overfitting (Train-Test)']
colors = ['green' if x < 0.1 else 'orange' if x < 0.2 else 'red' for x in overfitting]
ax2.bar(model_names, overfitting, color=colors, alpha=0.7)
ax2.set_xlabel('Models')
ax2.set_ylabel('Train - Test Accuracy')
ax2.set_title('Overfitting Analysis')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(True, alpha=0.3)

ax3 = axes[0, 2]
sorted_df = comparison_df.sort_values('Test Accuracy', ascending=True)
ax3.barh(sorted_df['Model'], sorted_df['Test Accuracy'], color='skyblue', alpha=0.8)
ax3.set_xlabel('Test Accuracy')
ax3.set_title('Model Ranking (Test Accuracy)')
ax3.grid(True, alpha=0.3)

for i, model_name in enumerate(model_names):
    row = 1 + (i // 3)
    col = i % 3
    
    if row < 3:  # Only plot if we have space in our 3x3 grid
        ax = axes[row, col]
        y_pred = results[model_name]['predictions']
        
        cm = confusion_matrix(y_test, y_pred)
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Non-Sarcastic', 'Sarcastic'],
                    yticklabels=['Non-Sarcastic', 'Sarcastic'])
        ax.set_title(f'{model_name}\nTest Acc: {results[model_name]["test_acc"]:.4f}')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()


In [ ]:
for model_name in model_names:
    y_pred = results[model_name]['predictions']
    cm = confusion_matrix(y_test, y_pred)
    
    tn, fp, fn, tp = cm.ravel()
    precision_0 = tn / (tn + fn) if (tn + fn) > 0 else 0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision_1 = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    print(f"\n{model_name}:")
    print(f"  Non-Sarcastic - Precision: {precision_0:.4f}, Recall: {recall_0:.4f}")
    print(f"  Sarcastic - Precision: {precision_1:.4f}, Recall: {recall_1:.4f}")

## 📝 Evaluation Criteria

Your homework will be evaluated based on:

1. **Implementation Correctness (40%)**
   - Proper text preprocessing pipeline
   - Correct TF-IDF vectorization
   - Working neural network architectures (2-Layer NN, RNN, GRU, LSTM)
   - Appropriate tokenization and vocabulary building for RNN models

2. **Model Performance (30%)**
   - Reasonable classification accuracy across all models
   - Proper convergence during training
   - Meaningful comparison between different approaches
   - Appropriate handling of text data and sequence modeling

3. **Code Quality and Analysis (30%)**
   - Clean, readable code with proper structure
   - Comprehensive model comparison and visualization
   - Insightful analysis of results and model differences
   - Good coding practices and efficient implementation


# Dataset Source
1. Misra, Rishabh and Prahal Arora. "Sarcasm Detection using News Headlines Dataset." AI Open (2023).
2. Misra, Rishabh and Jigyasa Grover. "Sculpting Data for ML: The first act of Machine Learning." ISBN 9798585463570 (2021).
